# Лабораторна 2. Від сирих даних до нейромережі

У цій лабораторній роботі ви пройдете повний цикл роботи з табличним датасетом для задачі регресії — від завантаження до порівняння моделей:

- дослідницький аналіз даних (EDA): загальний огляд, гістограми, кореляції;
- обробка пропусків і викидів;
- інженерія ознак;
- baseline: множинна лінійна регресія;
- нейромережа — підбір архітектури, функцій активації, оптимізатора власними експериментами;
- чесне порівняння моделей на test за метриками регресії (MSE, RMSE, MAE, R²).

Демо-ноутбук з лекції 3-4 — основа, звертайтесь до нього як до довідника, коли щось незрозуміло. Він лежить в архіві демо-матеріалів лекції 3 — посилання «Демо-матеріали (zip)» є в картці лекції на головній сторінці курсу.

**Виконав: Домбровський Богдан Володимирович**

In [ ]:
import sys, subprocess, importlib.util
for package in ['datasets','tensorboard','gradio','torch','scikit-learn','pandas','matplotlib']:
    module = {'scikit-learn':'sklearn'}.get(package,package)
    if importlib.util.find_spec(module) is None:
        subprocess.check_call([sys.executable,'-m','pip','install','-q',package])
import numpy as np, pandas as pd, torch, matplotlib.pyplot as plt
from torch import nn
from datasets import load_dataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from torch.utils.tensorboard import SummaryWriter
from pathlib import Path
import random, copy, shutil, os
SEED=42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.set_num_threads(2)
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
ROOT=Path.cwd() if Path.cwd().name=='lab2' else Path.cwd()/'lab2'
if 'google.colab' in sys.modules: ROOT=Path('/content/lab2')
MODELS_DIR=ROOT/'models'; RUNS_DIR=ROOT/'runs'
for d in [MODELS_DIR,RUNS_DIR]: d.mkdir(parents=True,exist_ok=True)
print(device, ROOT)

## 0. Опрацюйте демо-ноутбук з лекції
Прогляньте пояснення ноутбуку і запустіть всі клітинки демо-ноутбуку (крім тих секцій, в яких вказано, що їх запускати не можна).
Після цього дайте тут відповідь на запитання:
1. Чому для заповнення пропусків було обрано метод kNN?
2. Чому при тренуванні лінійної регресії train і val порівнюються лише на основі MSE, а після тренування на основі багатьох функцій? Проекспериментуйте в межах тренування з вибором функції і поділіться результатами.
3. Які висновки після клітинки 2.6 в демо Ви можете зробити?
4. Поясни для чого потрібен baseline в лінійній регресії і як він вираховується? Що є baseline для нейронної мережі?
5. Продемонструй дію всіх функцій активації на графіку функцій, згенерувавши 1000 випадкових значень Х і порахувавши для них Y через ці функції.
6. Яким чином в нейронній мережі утворилася така кількість параметрів, яка вказана після виконання 3.3?
7. Чи оптимально вибрано кількість епох в 3.5?

### 0.1. Мої відповіді
1. kNN використовує зв’язки між ознаками та схожими об’єктами, на відміну від заповнення однією медіаною. У демо окремий KNeighborsRegressor навчається для кожної колонки з пропусками лише на train.
2. MSE є диференційовною функцією втрат і однаковим критерієм для train/val. Після навчання RMSE показує помилку в одиницях таргета, MAE менш чутлива до великих помилок, R² показує якість відносно константного прогнозу. Нижче порівнюємо MSE та MAE як функції навчання на однаковому спліті демо-даних.
3. У 2.6 стандартизація дозволяє порівнювати абсолютні ваги; знак визначає напрям умовного впливу. Ваги diag_coord та bedperroom дозволяють оцінити внесок інженерії ознак, але не доводять причинність. Числовий результат відтворюється нижче.
4. Простий baseline прогнозує середнє значення таргета train для кожного об’єкта. Для нейромережі baseline — навчена множинна лінійна регресія.
5. Графіки чотирьох активацій наведено нижче.
6. У демо після інженерії залишаються 7 ознак: (7×64+64)+(64×32+32)+(32×1+1)=2625 параметрів. Лінійна модель має 8. ReLU не має навчуваних параметрів.
7. 200 епох не доведено оптимальними: у підсумку демо зазначено, що train і val ще спадають, без явного перенавчання. Варто продовжити навчання та обирати епоху за validation, а не test.

In [ ]:
x=torch.tensor(np.sort(np.random.default_rng(SEED).uniform(-8,8,1000)),dtype=torch.float32)
for name, fn in [('ReLU',nn.ReLU()),('Leaky ReLU',nn.LeakyReLU(.01)),('Sigmoid',nn.Sigmoid()),('Tanh',nn.Tanh())]:
    plt.plot(x.numpy(),fn(x).numpy(),label=name)
plt.legend(); plt.grid(); plt.xlabel('X'); plt.ylabel('Y'); plt.show()

In [ ]:
# Експеримент на датасеті демо (без заборонених клітинок публікації)
from sklearn.impute import KNNImputer
demo=load_dataset('fantkolja/california-housing-with-missing-data')['train'].to_pandas()
a,b=train_test_split(demo,test_size=.4,random_state=SEED); b,_=train_test_split(b,test_size=.5,random_state=SEED)
a=a.dropna(subset=['MedHouseVal']); b=b.dropna(subset=['MedHouseVal'])
cols=[k for k in a.select_dtypes(include='number').columns if k!='MedHouseVal']
prep=make_pipeline(StandardScaler(),KNNImputer(n_neighbors=5))
xa=prep.fit_transform(a[cols]); xb=prep.transform(b[cols])
A,B=[torch.tensor(v,dtype=torch.float32) for v in (xa,xb)]
Y,Z=[torch.tensor(v.to_numpy(),dtype=torch.float32).reshape(-1,1) for v in (a.MedHouseVal,b.MedHouseVal)]
for lossname,lossfn in [('MSE',nn.MSELoss()),('MAE',nn.L1Loss())]:
    torch.manual_seed(SEED); model=nn.Linear(A.shape[1],1); opt=torch.optim.SGD(model.parameters(),lr=.03)
    for epoch in range(100):
        opt.zero_grad(); loss=lossfn(model(A),Y); loss.backward(); opt.step()
    pred=model(B).detach().numpy().ravel(); print(lossname,'val MSE:',mean_squared_error(Z.numpy().ravel(),pred),'val MAE:',mean_absolute_error(Z.numpy().ravel(),pred))
    if lossname=='MSE': print(pd.Series(model.weight.detach().numpy().ravel(),index=cols).sort_values(key=abs,ascending=False))
print('Порівнюємо обидві функції за однаковими метриками; менша MAE не гарантує меншої MSE.')

---
## 1. Датасети

Ви отримаєте один з 10 датасетів нижче. Усі — задачі регресії на Hugging Face: табличні дані, здебільшого числові ознаки, від кількасот до кількадесяти тисяч рядків.

1. **[King County House Sales](https://huggingface.co/datasets/kraina/house_sales_in_king_county)** — 21 613 продажів будинків (Вашингтон, 2014–2015). Таргет: `price`. ⚠️ `load_dataset("kraina/house_sales_in_king_county")` не працює (застарілий loading-скрипт) — вантажте напряму: `load_dataset("parquet", data_files="https://huggingface.co/datasets/kraina/house_sales_in_king_county/resolve/main/data/house_sales_king_county.parquet")`.
2. **[Diamonds](https://huggingface.co/datasets/jdxcosta/diamonds)** — 53 940 діамантів. Таргет: `price`. Ознаки: вага (`carat`), огранка, колір, чистота, розміри.
3. **[Concrete Compressive Strength](https://huggingface.co/datasets/foundry-ml/dataset_concrete_compressive_strength)** — 1 030 замісів бетону. Таргет: міцність на стиск. Усі ознаки числові (склад суміші + вік).
4. **[Auto MPG](https://huggingface.co/datasets/scikit-learn/auto-mpg)** — 398 автомобілів. Таргет: `mpg` (витрата пального). Найменший і найпростіший датасет зі списку.
5. **[Wine Quality](https://huggingface.co/datasets/codesignal/wine-quality)** — 1 599 (червоне) + 4 898 (біле) вин. Таргет: `quality` (оцінка 3–9). Усі ознаки — хімічний склад, числові.
6. **[Laptop Price](https://huggingface.co/datasets/Ammok/laptop_price_prediction)** — 977 + 325 ноутбуків. Таргет: `Price`. Частина ознак текстові (`RAM`, `Storage`, `CPU`) — доведеться парсити в числа.
7. **[Bike Sharing Demand](https://huggingface.co/datasets/t22000t/bike-sharing-tabular)** — 17 379 годинних записів прокату велосипедів. Таргет: `cnt`. Часові й погодні ознаки.
8. **[Ames House Prices](https://huggingface.co/datasets/t22000t/house-prices-tabular)** — 1 460 будинків, 79 ознак. Таргет: `SalePrice`. Найбільше категоріальних ознак зі списку — гарна практика на encoding.
9. **[Medical Insurance Cost](https://huggingface.co/datasets/harishsohani/medical-insurance-cost-prediction)** — 1 338 застрахованих. Таргет: `charges`. Компактний, мало ознак — хороший старт, якщо часу мало. ⚠️ У репозиторії кілька CSV одразу (`Xtrain`/`Xtest`/`insurance.csv`) — вкажіть файл явно: `load_dataset("harishsohani/medical-insurance-cost-prediction", data_files="insurance.csv")`.
10. **[Abalone](https://huggingface.co/datasets/mstz/abalone)** — 4 177 молюсків. Таргет: `number_of_rings` (вік). Усі ознаки — фізичні виміри, окрім `sex`.

Решта (2–5, 7–8, 10) вантажаться напряму — `load_dataset("<repo_id>")` без додаткових аргументів, HF-токен не потрібен.

### 1.1. Який датасет мій?

Визначте самі власний датасет: беремо Ваше ПІБ, рахуємо SHA-256 хеш, залишок від ділення на 10 — це індекс у списку датасетів вище (0 → перший, 9 → останній).

**Формат ПІБ:** точно як у списку групи — `Прізвище Ім'я По-батькові`, по одному пробілу між словами, без зайвих пробілів на початку/в кінці. Якщо використаєте інший формат - **не зарахую лабу**!!!

In [ ]:
import hashlib

DATASETS = [
    "King County House Sales",
    "Diamonds",
    "Concrete Compressive Strength",
    "Auto MPG",
    "Wine Quality",
    "Laptop Price",
    "Bike Sharing Demand",
    "Ames House Prices",
    "Medical Insurance Cost",
    "Abalone",
]

FULL_NAME = "Домбровський Богдан Володимирович"  # TODO: впишіть своє ПІБ точно як у списку групи

dataset_index = int(hashlib.sha256(FULL_NAME.encode("utf-8")).hexdigest(), 16) % len(DATASETS)
print(f"Ваш датасет (#{dataset_index + 1}):", DATASETS[dataset_index])

## 2.1. Підготовка даних
Спліт 60/20/20 до EDA. Усі статистики препроцесингу обчислюються тільки на train, щоб уникнути витоку даних.

In [ ]:
ds=load_dataset('mstz/abalone')
df=pd.concat([v.to_pandas() for v in ds.values()],ignore_index=True)
TARGET='number_of_rings'
train,rest=train_test_split(df,test_size=.4,random_state=SEED)
val,test=train_test_split(rest,test_size=.5,random_state=SEED)
print('Розміри:',len(train),len(val),len(test)); display(train.head()); train.info(); display(train.describe(include='all')); print(train.isna().sum())
print('Пропуски:',int(train.isna().sum().sum()))
train.select_dtypes(include='number').hist(figsize=(13,9),bins=30); plt.tight_layout(); plt.show()
corr=train.select_dtypes(include='number').corr(); display(corr.round(2))
plt.figure(figsize=(9,7)); plt.imshow(corr,vmin=-1,vmax=1,cmap='coolwarm'); plt.xticks(range(len(corr)),corr.columns,rotation=90); plt.yticks(range(len(corr)),corr.columns); plt.colorbar(); plt.tight_layout(); plt.show()
print('Асиметрія:',train.select_dtypes(include='number').skew()); print('Найчастіші значення таргета:',train[TARGET].value_counts().head())

Фізичні розміри та маси можуть бути сильно корельованими, оскільки описують розмір одного молюска. Дискретний таргет не означає capping. Рідкісні великі значення можуть відповідати старшим особинам, тому не видаляємо їх без доказів помилки. Нульова висота фізично підозріла: замінюємо її на пропуск і імпутуємо медіаною train. Категорію sex кодуємо one-hot. Зберігаємо фізичні ознаки, щоб не втратити сигнал; додаємо співвідношення маси м’яса до загальної маси, якщо відповідні колонки наявні.

In [ ]:
raw_cols=[k for k in train.columns if k!=TARGET]
num_raw=[k for k in raw_cols if pd.api.types.is_numeric_dtype(train[k]) and k!='sex']
cat=[k for k in raw_cols if k not in num_raw]
def features(frame):
    f=frame[raw_cols].copy()
    for k in num_raw:
        if 'height' in k.lower(): f[k]=f[k].mask(f[k]<=0)
    whole=next((k for k in num_raw if 'whole' in k.lower() and 'weight' in k.lower()),None)
    shucked=next((k for k in num_raw if 'shucked' in k.lower() and 'weight' in k.lower()),None)
    if whole and shucked: f['meat_ratio']=f[shucked]/f[whole].where(f[whole]>0)
    for k in cat: f[k]=f[k].astype(object).where(f[k].notna(),np.nan)
    return f
F=features(train); nums=[k for k in F.columns if k not in cat]
pre=ColumnTransformer([('num',make_pipeline(SimpleImputer(strategy='median'),StandardScaler()),nums),('cat',make_pipeline(SimpleImputer(strategy='most_frequent'),OneHotEncoder(handle_unknown='ignore',sparse_output=False)),cat)])
X=[pre.fit_transform(F),pre.transform(features(val)),pre.transform(features(test))]
y=[d[TARGET].to_numpy(dtype=np.float32) for d in (train,val,test)]
xt,xv,xs=[torch.tensor(v,dtype=torch.float32,device=device) for v in X]
yt,yv,ys=[torch.tensor(v,dtype=torch.float32,device=device).reshape(-1,1) for v in y]
n_features=xt.shape[1]; print(pre.get_feature_names_out())

## 2.2–2.4. Лінійна регресія, архітектури та гіперпараметри
Однаковий спліт і seed; вибір архітектури та checkpoint за validation MSE. Три зміни окремо: learning rate, оптимізатор, епохи.

In [ ]:
models={}; histories={}; configs={}; bestvals={}
def fit(name,widths,lr,optname,epochs):
    torch.manual_seed(SEED)
    layers=[]; prev=n_features
    for w in widths: layers.extend([nn.Linear(prev,w),nn.ReLU()]); prev=w
    layers.append(nn.Linear(prev,1)); model=(nn.Sequential(*layers) if widths else layers[0]).to(device)
    opt=(torch.optim.Adam if optname=='Adam' else torch.optim.SGD)(model.parameters(),lr=lr)
    lossfn=nn.MSELoss(); writer=SummaryWriter(str(RUNS_DIR/name)); history=[]; best=float('inf')
    for e in range(epochs):
        model.train(); opt.zero_grad(); loss=lossfn(model(xt),yt); loss.backward(); opt.step()
        model.eval()
        with torch.no_grad(): tr=lossfn(model(xt),yt).item(); va=lossfn(model(xv),yv).item()
        history.append((tr,va)); writer.add_scalar('Loss/train',tr,e); writer.add_scalar('Loss/val',va,e)
        if va<best: best=va; state=copy.deepcopy(model.state_dict()); best_epoch=e+1
    writer.close(); model.load_state_dict(state); model.eval()
    models[name]=model; histories[name]=np.array(history); bestvals[name]=best; configs[name]=(widths,lr,optname,epochs)
    torch.save({k:v.cpu() for k,v in state.items()},MODELS_DIR/(name+'.pt'))
    print(name,'best val MSE',round(best,4),'epoch',best_epoch,'parameters',sum(p.numel() for p in model.parameters()))
fit('linear',[],.03,'SGD',400)
fit('nn_v1',[64,32],.01,'Adam',200)
fit('nn_v2',[128,64,32],.01,'Adam',200)
base=min(['nn_v1','nn_v2'],key=bestvals.get); widths,lr,opt,epochs=configs[base]
fit('nn_lr',widths,.001,opt,epochs)
fit('nn_optimizer',widths,lr,'SGD',epochs)
fit('nn_epochs',widths,lr,opt,400)
print('Архітектура для експериментів:',base)
for name,h in histories.items():
    plt.figure(figsize=(6,3)); plt.plot(h[:,0],label='train'); plt.plot(h[:,1],label='val'); plt.title(name); plt.xlabel('Epoch'); plt.ylabel('MSE'); plt.legend(); plt.show()
weights=pd.Series(models['linear'].weight.detach().cpu().numpy().ravel(),index=pre.get_feature_names_out()); display(weights.sort_values(key=abs,ascending=False))

## 2.5. Порівняння на test
Тестові метрики обчислюються після завершення всіх експериментів. Для one-hot ваги залежать від кодування, а корельовані ознаки не дозволяють робити причинні висновки.

In [ ]:
def metrics(pred):
    mse=mean_squared_error(y[2],pred); return dict(MSE=mse,RMSE=mse**.5,MAE=mean_absolute_error(y[2],pred),R2=r2_score(y[2],pred))
rows=[]
with torch.no_grad():
    for name,model in models.items(): rows.append(dict(model=name,**metrics(model(xs).cpu().numpy().ravel())))
results=pd.DataFrame(rows).sort_values('MSE'); results.to_csv(ROOT/'results.csv',index=False); display(results)
print('Константний baseline:',metrics(np.full(len(test),y[0].mean())))
best=results.iloc[0]; linear=results.set_index('model').loc['linear','MSE']; nn_best=results[results.model!='linear'].iloc[0]
print(f"Найменша test MSE: {best.model}, {best.MSE:.4f}. Найкраща NN змінює MSE відносно linear на {(linear-nn_best.MSE)/linear*100:.2f}% (додатне — покращення).")
print('Модель, обрана за validation:',min(bestvals,key=bestvals.get))
changes={'архітектура':bestvals['nn_v1']-bestvals['nn_v2'],**{k:bestvals[base]-bestvals[k] for k in ['nn_lr','nn_optimizer','nn_epochs']}}
print('Зміни validation MSE (додатне — покращення):',changes); print('Найбільший ефект за модулем:',max(changes,key=lambda k:abs(changes[k])))
for name,h in histories.items(): plt.plot(h[:,1],label=name)
plt.legend(); plt.xlabel('Epoch'); plt.ylabel('Validation MSE'); plt.show()

## 3. Gradio — порівняння всіх моделей наживо

In [ ]:
import gradio as gr
inputs=[]
for k in raw_cols:
    if k in cat: inputs.append(gr.Dropdown(choices=train[k].dropna().unique().tolist(),value=train[k].mode().iloc[0],label=k))
    else: inputs.append(gr.Number(value=float(train[k].median()),label=k))
def predict_all(*values):
    row=pd.DataFrame([values],columns=raw_cols); t=torch.tensor(pre.transform(features(row)),dtype=torch.float32,device=device)
    with torch.no_grad(): return {name:float(model(t).item()) for name,model in models.items()}
app=gr.Interface(fn=predict_all,inputs=inputs,outputs=gr.JSON(),title='Abalone: порівняння моделей')
app.launch(share=False,inline=True,prevent_thread_lock=True)

## 4. Збереження результатів
Архів містить моделі, логи та CSV. Виконаний ноутбук завантажте окремо через File → Download → Download .ipynb та додайте до lab2.

In [ ]:
archive=shutil.make_archive(str(ROOT.parent/'lab2_artifacts'),'zip',ROOT.parent,ROOT.name)
print(archive)
if 'google.colab' in sys.modules:
    from google.colab import files
    files.download(archive)

---
### Як здавати лабу

У форму (https://forms.gle/8Dbc9zB8L5tsNuNEA) здати посилання на **публічний** репозиторій, який містить:

1. **Цей ноутбук з усіма виконаними пунктами**. Має запускатися зверху до низу (Run All) в будь-якому середовищі без помилок.
2. **Файли всіх натренованих варіантів моделі** (2.2-2.4) — лінійна регресія + всі варіанти нейромережі, кожен окремим `.pt` файлом.
3. **Логи TensorBoard** — папка `runs/` з усіма прогонами, щоб можна було відкрити криві навчання, не перезапускаючи ноутбук.
4. **Таблицю результатів** — `results.csv` з порівнянням усіх варіантів (модель, MSE, RMSE, MAE, R² на test).
5. Структура файлів на репозиторії, **точно дотримуйтесь, інакше не зарахується лаба**:
```
/lab2/nn_task.ipynb
/lab2/models/linear.pt
/lab2/models/nn_v1.pt
/lab2/models/nn_v2.pt
/lab2/models/...        (усі інші варіанти з 2.3-2.4)
/lab2/runs/*
/lab2/results.csv
```